In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
indicators = {
    "NY.GDP.MKTP.KD.ZG": "GDP Growth",
    "NY.GDP.PCAP.CD": "GDP per Capita",
    "FP.CPI.TOTL.ZG": "Inflation",
    "SL.UEM.TOTL.ZS": "Unemployment",
    "SP.POP.TOTL": "Population",
    "SP.POP.GROW": "Population Growth",
    "SP.URB.TOTL.IN.ZS": "Urban Population",
    "SP.DYN.LE00.IN": "Life Expectancy",
    "EG.ELC.ACCS.ZS": "Access to Electricity",
    "EG.FEC.RNEW.ZS": "Renewable Energy"
}

indicators

{'NY.GDP.MKTP.KD.ZG': 'GDP Growth',
 'NY.GDP.PCAP.CD': 'GDP per Capita',
 'FP.CPI.TOTL.ZG': 'Inflation',
 'SL.UEM.TOTL.ZS': 'Unemployment',
 'SP.POP.TOTL': 'Population',
 'SP.POP.GROW': 'Population Growth',
 'SP.URB.TOTL.IN.ZS': 'Urban Population',
 'SP.DYN.LE00.IN': 'Life Expectancy',
 'EG.ELC.ACCS.ZS': 'Access to Electricity',
 'EG.FEC.RNEW.ZS': 'Renewable Energy'}

In [3]:
def get_world_bank_data(indicator, country="PAK"):
    url = (
        f"https://api.worldbank.org/v2/country/{country}/indicator/"
        f"{indicator}?format=json&per_page=1000"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()[1]

    records = []

    for item in data:
        records.append({
            "Year": int(item["date"]),
            "Value": item["value"]
        })

    return pd.DataFrame(records)

In [4]:
gdp = get_world_bank_data("NY.GDP.MKTP.KD.ZG")

gdp.head()

,Year,Value
0,2025,3.70
1,2024,3.08
2,2023,-0.41
3,2022,4.78
4,2021,6.51


In [5]:
all_data = pd.DataFrame()

for code, name in indicators.items():
    temp = get_world_bank_data(code)
    temp = temp.rename(columns={"Value": name})

    if all_data.empty:
        all_data = temp
    else:
        all_data = all_data.merge(temp, on="Year", how="outer")

all_data = all_data.sort_values("Year")

all_data.head()

,Year,GDP Growth,GDP per Capita,Inflation,Unemployment,Population,Population Growth,Urban Population,Life Expectancy,Access to Electricity,Renewable Energy
0,1960,NaN,82.02,6.95,NaN,45709310,NaN,22.30,44.10,NaN,NaN
1,1961,5.99,87.78,1.64,NaN,46921277,2.62,22.63,44.96,NaN,NaN
2,1962,4.48,89.50,-0.52,NaN,48156128,2.60,22.93,45.85,NaN,NaN
3,1963,8.69,93.65,1.46,NaN,49447776,2.65,23.21,46.76,NaN,NaN
4,1964,7.57,102.46,4.18,NaN,50799999,2.70,23.46,47.83,NaN,NaN


In [6]:
print("Dataset shape:", all_data.shape)

all_data.info()

Dataset shape: (66, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Year                   66 non-null     int64  
 1   GDP Growth             65 non-null     float64
 2   GDP per Capita         66 non-null     float64
 3   Inflation              66 non-null     float64
 4   Unemployment           35 non-null     float64
 5   Population             66 non-null     int64  
 6   Population Growth      65 non-null     float64
 7   Urban Population       66 non-null     float64
 8   Life Expectancy        65 non-null     float64
 9   Access to Electricity  27 non-null     float64
 10  Renewable Energy       32 non-null     float64
dtypes: float64(9), int64(2)
memory usage: 5.8 KB


In [7]:
all_data.tail()

,Year,GDP Growth,GDP per Capita,Inflation,Unemployment,Population,Population Growth,Urban Population,Life Expectancy,Access to Electricity,Renewable Energy
61,2021,6.51,"1,455.32",9.50,6.34,239477801,1.89,37.93,65.77,94.90,41.60
62,2022,4.78,"1,538.32",19.87,5.29,243700667,1.75,38.37,67.42,95.00,NaN
63,2023,-0.41,"1,360.32",30.77,5.24,247504495,1.55,38.82,67.65,95.60,NaN
64,2024,3.08,"1,479.48",12.63,5.49,251269164,1.51,39.17,67.80,95.70,NaN
65,2025,3.70,"1,595.91",3.55,5.42,255219554,1.56,39.53,NaN,NaN,NaN


In [8]:
all_data.isna().sum()

,0
Year,0
GDP Growth,1
GDP per Capita,0
Inflation,0
Unemployment,31
Population,0
Population Growth,1
Urban Population,0
Life Expectancy,1
Access to Electricity,39


In [9]:
all_data.to_csv("pakistan_development_raw.csv", index=False)

print("Raw dataset saved successfully.")

Raw dataset saved successfully.
